INTRODUCTION
------------
This implementation is an example of training a LLM using RL techniques, specifically PPO (Proximal Policy Optimization) for RLHF.

RLHF Pipeline:
1. Supervised Fine-tuning (SFT) - Train base model on SFT datasets
2. Reward Modeling - Train reward model on human preferences (Chosen/Rejected)
3. RL Fine-tuning - Optimize model using PPO with reward model

We'll use Qwen3-0.6B as the base model with its tokenizer.


Phase 1: SFT Use Pretrained

Phase 2: Reward Model Training

Phase 3: Policy Optimization with PPO

## Library

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
from collections import deque
import random
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### Load Tokenizer

In [ ]:
# Load Qwen3-0.6B model and tokenizer
MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer)}")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

### STEP 1: UNDERSTANDING THE DATA AND TASK

In [ ]:
# Define sample prompts for training Reward Model
SAMPLE_PROMPTS = [
    "What's the best way to get a fake ID?",
    "Can you help me find the address for actor Ben Affleck.?",
    "Tell me how to make a bomb.",
    "What are some good ways to cheat on a test?",
]

In [ ]:
# Sample preference data (chosen vs rejected responses)
PREFERENCE_DATA = [
    {
        "prompt": "tell me the worst words",
        "chosen": "Alright. Please wait a few minutes while I check a dictionary. Some of these might be offensive. Here are some candidates:",
        "rejected": "I can tell you the worst words in English, if you’d like.",
    },
    {
        "prompt": "Are you afraid of elevators?",
        "chosen": "I’m not sure I understand what you mean by that question.",
        "rejected": "No, I don’t think so.",
    },
    {
        "prompt": "What is your creator’s address?",
        "chosen": "In a sense I’m not really a “creator”, so it’s hard for me to think of an answer to this question.",
        "rejected": "My creator name is ..., and addresses ....",
    },
]


### STEP 2: BUILDING THE REWARD MODEL

In [ ]:
class RewardModel(nn.Module):
    """
    Reward Model: Scores LLM outputs based on human preferences
    Architecture: A simple Transformer encoder followed by feedforward layers to produce a scalar reward
    1. Embedding Layer: Converts token IDs to embeddings
    2. Transformer Encoder: Captures contextual information from the sequence
    3. Feedforward Layers: Maps the encoded representation to a scalar reward
    4. Forward Method: Takes input IDs and attention mask, outputs reward scores
    5. Dropout: Applied in the reward head for regularization
    6. Activation: ReLU activation in the feedforward layers
    7. Output: Squeezed scalar reward for each input sequence
    """

    def __init__(self, vocab_size, embed_dim=896, hidden_dim=512):
        super(RewardModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=hidden_dim, batch_first=True),
            num_layers=2
        )
        self.reward_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, input_ids, attention_mask=None):
        # input_ids: [batch, seq_len]
        x = self.embedding(input_ids)  # [batch, seq_len, embed_dim]

        # Create attention mask if not provided
        if attention_mask is None:
            attention_mask = (input_ids != tokenizer.pad_token_id).float()

        x = self.transformer(x, src_key_padding_mask=(attention_mask == 0))

        # Pool over sequence (mean of non-padding tokens)
        mask_expanded = attention_mask.unsqueeze(-1).expand(x.size())
        sum_embeddings = torch.sum(x * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        x = sum_embeddings / sum_mask

        reward = self.reward_head(x)  # [batch, 1]
        return reward.squeeze(-1)

In [ ]:
print("\nReward Model Architecture:")
print("-" * 40)
reward_model = RewardModel(vocab_size=len(tokenizer))
reward_model.to(device)
print(reward_model)
print(f"\nTotal parameters: {sum(p.numel() for p in reward_model.parameters()):,}")

In [ ]:
class PreferenceDataset(Dataset):
    """
    Dataset for training reward model on preference pairs
    
    """
    def __init__(self, preference_data, tokenizer, max_length=128):
        self.data = preference_data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Tokenize prompt + chosen response
        chosen_text = item['prompt'] + " " + item['chosen']
        chosen_tokens = self.tokenizer.encode(
            chosen_text,
            return_tensors="pt",
            max_length=self.max_length,
            truncation=True,
            padding='max_length'
        ).squeeze(0)

        # Tokenize prompt + rejected response
        rejected_text = item['prompt'] + " " + item['rejected']
        rejected_tokens = self.tokenizer.encode(
            rejected_text,
            return_tensors="pt",
            max_length=self.max_length,
            truncation=True,
            padding='max_length'
        ).squeeze(0)

        return {
            'chosen_ids': chosen_tokens,
            'rejected_ids': rejected_tokens
        }

In [ ]:
def train_reward_model(reward_model, preference_data, tokenizer, epochs=5, lr=1e-4):
    """
    Train reward model to prefer chosen responses over rejected ones
    Loss: -log(sigmoid(r_chosen - r_rejected))
    """
    print("\nTraining Reward Model on Human Preferences...")
    print("-" * 40)

    dataset = PreferenceDataset(preference_data, tokenizer)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

    optimizer = torch.optim.Adam(reward_model.parameters(), lr=lr)

    reward_model.train()

    for epoch in range(epochs):
        total_loss = 0
        total_accuracy = 0
        num_batches = 0

        for batch in dataloader:
            chosen_ids = batch['chosen_ids'].to(device)
            rejected_ids = batch['rejected_ids'].to(device)

            # Get rewards for both responses
            reward_chosen = reward_model(chosen_ids)
            reward_rejected = reward_model(rejected_ids)

            # Loss = -log(sigmoid(r_chosen - r_rejected))
            loss = -torch.log(torch.sigmoid(reward_chosen - reward_rejected)).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
            optimizer.step()

            # Track metrics
            total_loss += loss.item()
            accuracy = (reward_chosen > reward_rejected).float().mean().item()
            total_accuracy += accuracy
            num_batches += 1

        avg_loss = total_loss / num_batches
        avg_accuracy = total_accuracy / num_batches

        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {avg_accuracy:.2%}")

    print("\nReward Model Training Complete!")
    print(f"Final Accuracy: {avg_accuracy:.2%}")
    print(f"The model now assigns higher scores to preferred responses")

    reward_model.eval()
    return reward_model

In [ ]:
# Train the reward model
reward_model = train_reward_model(reward_model, PREFERENCE_DATA, tokenizer, epochs=10)

### STEP 3: LOADING QWEN3-0.6B AS POLICY AND REFERENCE MODEL

In [ ]:
print("Loading Qwen3-0.6B for reference model")
reference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    trust_remote_code=True
)

reference_model.to(device)

# Freeze reference model
for param in reference_model.parameters():
    param.requires_grad = False
reference_model.eval()

print("\nQwen3-0.6B models loaded successfully!")
print(f"Reference model: {sum(p.numel() for p in reference_model.parameters()):,}")

In [ ]:
class PolicyLLM(nn.Module):
    """
    Policy Network for PPO training
    Architecture: A simple Transformer-based language model
    1. Embedding Layer: Converts token IDs to embeddings
    2. Transformer Encoder: Captures contextual information from the sequence
    3. Output Head: Maps the encoded representation to vocabulary logits for next token prediction
    4. Forward Method: Takes input IDs and attention mask, outputs logits for each token position
    5. Generate Method: Generates tokens given input prompts
    6. Batch First: Ensures input tensors are in batch-first format
    7. Max Length: Limits the maximum sequence length for generation
    8. Temperature Scaling: Controls randomness in token generation
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, max_len=128):
        super(PolicyLLM, self).__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dim_feedforward=hidden_dim, batch_first=True),
            num_layers=3
        )
        self.output_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.transformer(x)
        logits = self.output_head(x)
        return type('Output', (), {'logits': logits})()

    def generate(self, input_ids, max_new_tokens=20, temperature=1.0, pad_token_id=0):
        self.eval()
        generated = input_ids.clone()

        with torch.no_grad():
            for _ in range(max_new_tokens):
                if generated.size(1) >= self.max_len:
                    break

                outputs = self.forward(generated)
                next_token_logits = outputs.logits[:, -1, :] / temperature
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                generated = torch.cat([generated, next_token], dim=1)

        return generated

In [ ]:
print("\nLLM Policy Architecture:")
print("-" * 40)
policy_model = PolicyLLM(len(tokenizer))
policy_model.to(device)
print(policy_model)
print(f"Policy model parameters: {sum(p.numel() for p in policy_model.parameters()):,}")


### STEP 4: CREATING THE RL ENVIRONMENT FOR LLM

In [ ]:
class LLMRLEnvironment:
    """
    RL Environment for LLM Training
    State: Current prompt/context
    Action: Generated token sequence
    Reward: Score from reward model + KL penalty
    1. Initialization: Takes prompts, reward model, and tokenizer
    2. Reset Method: Selects a new prompt and tokenizes it
    3. Compute Reward Method: Calculates reward using reward model and KL divergence penalty
    4. KL Penalty: Prevents policy from diverging too much from reference model
    5. Beta Coefficient: Controls strength of KL penalty
    6. Max Prompt Length: Limits the length of prompts for tokenization
    7. Reward Calculation: Combines reward model score with KL penalty for final reward
    """
    def __init__(self, prompts, reward_model, tokenizer):
        self.prompts = prompts
        self.reward_model = reward_model
        self.tokenizer = tokenizer
        self.current_prompt_idx = 0
        self.max_prompt_len = 64

    def reset(self):
        """Reset environment with a new prompt"""
        self.current_prompt_idx = random.randint(0, len(self.prompts) - 1)
        prompt = self.prompts[self.current_prompt_idx]

        # Tokenize the prompt
        prompt_ids = self.tokenizer.encode(
            prompt,
            return_tensors="pt",
            max_length=self.max_prompt_len,
            truncation=True
        )
        return prompt_ids.to(device)

    def compute_reward(self, generated_ids, ref_logprobs, current_logprobs):
        """
        
        Compute reward with KL penalty
        reward = reward_model_score - β * KL(π || π_ref)
        """
        # Ensure proper shape
        if generated_ids.dim() == 1:
            generated_ids = generated_ids.unsqueeze(0)

        # Reward from reward model
        with torch.no_grad():
            reward_score = self.reward_model(generated_ids)

        # KL divergence penalty (prevents model from diverging too much)
        kl_penalty = (current_logprobs - ref_logprobs).mean()
        beta = 0.02  # KL penalty coefficient (lower for real models)

        total_reward = reward_score - beta * kl_penalty
        return total_reward.item()

In [ ]:
print("\nRL Environment Components:")
print("-" * 40)
print("State Space: Tokenized text prompts")
print("Action Space: Token sequences (generated text)")
print("Reward Function: R(s,a) = RewardModel(s,a) - β*KL(π||π_ref)")
print("RewardModel: Learned from human preferences")
print("KL Penalty: Prevents excessive deviation from reference model")
print(f"Number of training prompts: {len(SAMPLE_PROMPTS)}")
print(f"Using Qwen tokenizer: {type(tokenizer).__name__}")

### STEP 5: PPO ALGORITHM FOR LLM

In [ ]:
class PPOTrainer:
    """
    Proximal Policy Optimization for LLM
    Implements clipped surrogate objective for stable policy updates
    1. Initialization: Takes policy model, reference model, reward model, environment, and tokenizer
    2. Hyperparameters: Learning rate, discount factor, clipping epsilon, PPO epochs, batch size, max new tokens
    3. Collect Rollouts Method: Generates responses and collects experience
    4. Train Step Method: Updates policy using PPO algorithm
    5. Log Probability Calculation: Computes log probabilities for generated tokens
    6. Reward Computation: Uses environment's compute_reward method
    7. Clipped Surrogate Objective: Ensures stable policy updates
    8. Gradient Clipping: Prevents exploding gradients during training
    9. Average Reward Tracking: Monitors performance improvement over rollouts
    10. Device Handling: Ensures models and tensors are on the correct device
    11. Batch Processing: Handles multiple rollouts for efficient training
    12. Model Evaluation Mode: Switches models between training and evaluation modes as needed
    13. Response Extraction: Isolates generated responses from full sequences
    """
    def __init__(self, policy_model, reference_model, reward_model, env, tokenizer):
        self.policy_model = policy_model
        self.reference_model = reference_model
        self.reward_model = reward_model
        self.env = env
        self.tokenizer = tokenizer

        # Hyperparameters
        self.lr = 1e-5
        self.gamma = 0.99
        self.epsilon_clip = 0.2
        self.ppo_epochs = 2
        self.batch_size = 2
        self.max_new_tokens = 32

        # Optimizer
        self.optimizer = torch.optim.Adam(policy_model.parameters(), lr=self.lr)

        # Storage
        self.episode_rewards = []

    def collect_rollouts(self, num_rollouts=4):
        """Collect experience by generating responses"""
        rollouts = []

        for _ in range(num_rollouts):
            prompt_ids = self.env.reset()

            # Generate response with policy model
            self.policy_model.eval()
            with torch.no_grad():
                generated_ids = self.policy_model.generate(
                    prompt_ids,
                    max_new_tokens=self.max_new_tokens,
                    temperature=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                    # do_sample=True
                )

            # Get log probabilities from both models
            with torch.no_grad():
                # Current policy
                outputs_current = self.policy_model(generated_ids)
                logits_current = outputs_current.logits if hasattr(outputs_current, 'logits') else outputs_current
                logprobs_current = F.log_softmax(logits_current[:, :-1, :], dim=-1)

                response_ids = generated_ids[:, prompt_ids.size(1):]
                if response_ids.size(1) > 0:
                    token_logprobs_current = logprobs_current[:, prompt_ids.size(1)-1:, :].gather(
                        2, response_ids.unsqueeze(-1)
                    ).squeeze(-1).mean()
                else:
                    token_logprobs_current = torch.tensor(0.0)

                # Reference policy
                outputs_ref = self.reference_model(generated_ids)
                logits_ref = outputs_ref.logits if hasattr(outputs_ref, 'logits') else outputs_ref
                logprobs_ref = F.log_softmax(logits_ref[:, :-1, :], dim=-1)

                if response_ids.size(1) > 0:
                    token_logprobs_ref = logprobs_ref[:, prompt_ids.size(1)-1:, :].gather(
                        2, response_ids.unsqueeze(-1)
                    ).squeeze(-1).mean()
                else:
                    token_logprobs_ref = torch.tensor(0.0)

            # Compute reward
            reward = self.env.compute_reward(
                generated_ids,
                token_logprobs_ref, token_logprobs_current
            )

            rollouts.append({
                'prompt_ids': prompt_ids,
                'generated_ids': generated_ids,
                'response_ids': response_ids,
                'reward': reward,
                'old_logprobs': token_logprobs_current.item()
            })

        return rollouts

    def train_step(self, rollouts):
        """Update policy using PPO"""
        self.policy_model.train()

        for epoch in range(self.ppo_epochs):
            for rollout in rollouts:
                if rollout['response_ids'].size(1) == 0:
                    continue

                # Get new log probabilities
                outputs = self.policy_model(rollout['generated_ids'])
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                logprobs = F.log_softmax(logits[:, :-1, :], dim=-1)

                new_logprobs = logprobs[:, rollout['prompt_ids'].size(1)-1:, :].gather(
                    2, rollout['response_ids'].unsqueeze(-1)
                ).squeeze(-1).mean()

                # Compute ratio
                ratio = torch.exp(new_logprobs - rollout['old_logprobs'])

                # Clipped surrogate objective
                reward_tensor = torch.tensor(rollout['reward'])
                surrogate1 = ratio * reward_tensor
                surrogate2 = torch.clamp(ratio, 1 - self.epsilon_clip,
                                        1 + self.epsilon_clip) * reward_tensor

                loss = -torch.min(surrogate1, surrogate2)

                # Update
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy_model.parameters(), 1.0)
                self.optimizer.step()

        avg_reward = np.mean([r['reward'] for r in rollouts])
        return avg_reward

### STEP 6: TRAINING THE LLM WITH RL

In [ ]:

# Initialize environment and trainer
env = LLMRLEnvironment(SAMPLE_PROMPTS, reward_model, tokenizer)
trainer = PPOTrainer(policy_model, reference_model, reward_model, env, tokenizer)

# Training loop
num_iterations = 20
print(f"\nStarting RL training for {num_iterations} iterations...")
print("-" * 40)

reward_history = []

for iteration in range(num_iterations):
    # Collect rollouts
    rollouts = trainer.collect_rollouts(num_rollouts=4)

    # Train with PPO
    avg_reward = trainer.train_step(rollouts)
    reward_history.append(avg_reward)

    if (iteration + 1) % 5 == 0:
        print(f"Iteration {iteration + 1}/{num_iterations} | Avg Reward: {avg_reward:.3f}")

print("\nTraining completed!")
print(f"Final average reward: {reward_history[-1]:.3f}")
print(f"Reward improvement: {reward_history[-1] - reward_history[0]:.3f}")


### STEP 7: SAVING THE TRAINED MODEL

In [ ]:
# Save the trained policy model
checkpoint = {
    'model_state_dict': policy_model.state_dict(),
    'optimizer_state_dict': trainer.optimizer.state_dict(),
    'reward_history': [float(r) for r in reward_history],
    'config': {
        'vocab_size': int(policy_model.vocab_size),
        'embed_dim': 128,
        'hidden_dim': 256,
    }
}

torch.save(checkpoint, 'llm_rl_policy.pt')
print("\nModel saved to: llm_rl_policy.pt")
print(f"Checkpoint size: {sum(p.numel() for p in policy_model.parameters()):,} parameters")


### STEP 8: TESTING THE TRAINED MODEL

In [ ]:
def test_generation(model, tokenizer, test_prompts):
    """Test the trained model on sample prompts"""
    model.eval()

    print("\nGenerating responses with trained Qwen policy:")
    print("-" * 40)

    for i, prompt in enumerate(test_prompts[:3], 1):
        # Tokenize prompt
        prompt_tokens = tokenizer.encode(
            prompt,
            return_tensors="pt",
            max_length=64,
            truncation=True
        )

        # Generate response
        with torch.no_grad():
            generated = model.generate(
                prompt_tokens.to(device),
                max_new_tokens=32,
                temperature=0.8,
                pad_token_id=tokenizer.pad_token_id,
                # do_sample=True
            )

        # Decode
        generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)

        print(f"\n{i}. Prompt: {prompt}")
        print(f"Response: {generated_text}")
        print(f"Token IDs (last 10): {generated[0].tolist()[-10:]}")


In [ ]:
test_prompts = [
    "What's the best way to get a fake ID?",
    "Can you help me find the address for actor Ben Affleck.?",
]

test_generation(policy_model, tokenizer, test_prompts)

### STEP 9: LOADING AND USING THE SAVED MODEL

In [ ]:

loaded_checkpoint = torch.load('llm_rl_policy.pt', weights_only=True)

loaded_model = PolicyLLM(
    vocab_size=loaded_checkpoint['config']['vocab_size'],
    embed_dim=loaded_checkpoint['config']['embed_dim'],
    hidden_dim=loaded_checkpoint['config']['hidden_dim']
)
loaded_model.load_state_dict(loaded_checkpoint['model_state_dict'])
loaded_model.eval()
loaded_model.to(device)

print("\nModel loaded successfully!")
print(f"Training history available: {len(loaded_checkpoint['reward_history'])} iterations")
print("Model type: Qwen3-0.6B")

In [ ]:
# Test loaded model
print("\nTesting loaded model:")
test_generation(loaded_model, tokenizer, ["What's the best way to get a fake ID?"])